# Notebook 01 — Existing Model Analysis

Inspect `pavanmantha/mednli_laya` before writing any new code.

Goals:
1. Load the model and understand its interface
2. Test each LAYA primitive (choice, yesno, score) separately
3. Identify failure modes and edge cases
4. Establish baseline metrics on MedNLI test set
5. Document limitations

In [ ]:
import sys, json
sys.path.insert(0, '..')
import laya
import torch
print('laya version:', laya.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
# Load the existing fine-tuned model
MODEL = 'pavanmantha/mednli_laya'
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
agent = laya.load(MODEL, device=DEVICE)
print('Model loaded on', DEVICE)

In [ ]:
# --- 1. Test the choice (verdict) question ---
verdict_q = {
    'type': 'choice',
    'instructions': 'Decide whether the cited text supports the claim.',
    'criteria': {
        'supported': 'the cited text confirms the claim',
        'unsupported': 'the cited text does not say enough to confirm or deny the claim',
        'contradicted': 'the cited text states the opposite of the claim',
    }
}
questions = {'verdict': verdict_q}

test_cases = [
    # (claim, evidence, expected)
    ('Patient has elevated creatinine.',
     'Labs were notable for Cr 1.7 (baseline 0.5).',
     'supported'),
    ('Hemoglobin was within normal limits.',
     'Patient was found to have Hb of 7.2 g/dL.',
     'contradicted'),
    ('Patient was transferred from another hospital.',
     'Patient presents with 3 days of shortness of breath.',
     'unsupported'),
]

for claim, evidence, expected in test_cases:
    state = {'claim': claim, 'evidence': evidence}
    res = agent.predict(state, questions)['answers']['verdict']
    icon = '✓' if res['answer'] == expected else '✗'
    print(f"{icon} [{res['answer']} {res['probabilities'][res['answer']]:.2%}] expected={expected}")
    print(f'   {claim[:60]}...')

In [ ]:
# --- 2. Test yesno primitive ---
yesno_q = {
    'type': 'yesno',
    'instructions': 'Does the evidence explicitly mention the topic of the claim?',
    'criteria': {
        'yes': 'the evidence directly addresses the claim topic',
        'no': 'the evidence does not mention the claim topic',
    }
}
questions_yn = {'explicitly_mentioned': yesno_q}

for claim, evidence, expected in test_cases:
    state = {'claim': claim, 'evidence': evidence}
    res = agent.predict(state, questions_yn)['answers']['explicitly_mentioned']
    print(f"  explicitly_mentioned={res.get('answer', 'N/A')}  probs={res.get('probabilities', {})}")

In [ ]:
# --- 3. Test score primitive ---
score_q = {
    'type': 'score',
    'instructions': 'Rate the sufficiency of the evidence for evaluating the claim.',
    'criteria': {
        'min': 'the evidence provides no information relevant to the claim',
        'max': 'the evidence is complete and directly settles the claim',
    }
}
questions_sc = {'evidence_sufficiency': score_q}

for claim, evidence, expected in test_cases:
    state = {'claim': claim, 'evidence': evidence}
    res = agent.predict(state, questions_sc)['answers']['evidence_sufficiency']
    print(f"  evidence_sufficiency={res.get('score', res.get('answer', 'N/A')):.3f}  expected_verdict={expected}")

In [ ]:
# --- 4. Question wording sensitivity test ---
wordings = [
    {'type': 'choice', 'instructions': 'Check the claim against the evidence.',
     'criteria': {'supported': 'evidence establishes claim is true',
                  'unsupported': 'evidence neither establishes nor rules out claim',
                  'contradicted': 'evidence rules out claim'}},
    {'type': 'choice', 'instructions': 'Is the claim backed by the clinical evidence?',
     'criteria': {'supported': 'yes, evidence backs the claim',
                  'unsupported': 'evidence is not enough to decide',
                  'contradicted': 'no, evidence shows claim is false'}},
]

claim = 'Patient has elevated creatinine.'
evidence = 'Labs were notable for Cr 1.7 (baseline 0.5).'
state = {'claim': claim, 'evidence': evidence}

for i, wording in enumerate(wordings):
    res = agent.predict(state, {'verdict': wording})['answers']['verdict']
    print(f"Wording {i+1}: {res['answer']} ({res['probabilities'][res['answer']]:.2%})")

In [ ]:
# --- 5. Edge cases ---
edge_cases = [
    ('Patient is healthy.', 'Patient is healthy.', 'identical claim and evidence'),
    ('Patient has diabetes.', '', 'empty evidence'),
    ('X', 'The patient presented with severe chest pain and was found to have elevated troponin levels '
          'consistent with NSTEMI. Echo showed EF of 35%. Started on heparin drip. '
          'Cardiology was consulted and recommended cardiac catheterization.',
     'very short claim, long evidence'),
]

for claim, evidence, desc in edge_cases:
    try:
        state = {'claim': claim, 'evidence': evidence}
        res = agent.predict(state, {'verdict': verdict_q})['answers']['verdict']
        print(f"[{desc}] → {res['answer']} ({res['probabilities'][res['answer']]:.2%})")
    except Exception as e:
        print(f"[{desc}] ERROR: {e}")

## Summary of findings

Document your observations here:
- Which primitives work reliably?
- What are the failure modes?
- How sensitive is the model to question wording?
- What edge cases cause problems?
